# MCQ Generator using LangChain + Groq

Reads a local PDF, splits it into chunks, and generates multiple-choice questions
from each chunk using Groq's Llama model.

In [ ]:
!pip install -q langchain langchain-openai langchain-groq langchain-text-splitters pymupdf

## Set Groq API Key

Set your key as an environment variable before running:
- **Windows:** `set GROQ_API_KEY=your_key_here` in terminal, then launch VS Code from that terminal
- **Mac/Linux:** `export GROQ_API_KEY=your_key_here`

Or create a `.env` file in the same folder as this notebook with:
```
GROQ_API_KEY=your_key_here
```

In [ ]:
import os

# Load from .env file if present
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # dotenv not installed, rely on environment variable

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not set. See the instructions in the cell above.")

print("API key loaded.")

## Step 1 — Load PDF

Set `PDF_PATH` to the path of your local PDF file.

In [ ]:
import fitz  # PyMuPDF

PDF_PATH = "your_document.pdf"  # <-- change this to your PDF file path

doc = fitz.open(PDF_PATH)
text = ""
for page in doc:
    text += page.get_text()

print(f"Extracted {len(text)} characters from {len(doc)} pages")
print(text[:500])

## Step 2 — Split into Chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_text(text)
print(f"Total chunks: {len(chunks)}")
print("\nSample chunk:\n", chunks[1] if len(chunks) > 1 else chunks[0])

## Step 3 — Set Up LLM and Prompt

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile"
)

prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Generate 5 MCQ questions from the text below.

TEXT:
{text}

Format each question exactly as:
Question: <question>
A. <option>
B. <option>
C. <option>
D. <option>
Correct Answer: <letter>. <answer>
"""
)

## Step 4 — Generate MCQs from First Chunk

In [ ]:
chain = prompt | llm

response = chain.invoke({"text": chunks[0]})
print(response.content)

## Step 5 — Batch Generate MCQs for Multiple Chunks

In [ ]:
# Generate MCQs for the first 3 chunks (or fewer if the PDF is short)
num_chunks = min(3, len(chunks))

for i in range(num_chunks):
    print(f"\n{'='*60}")
    print(f"CHUNK {i+1} MCQs")
    print('='*60)
    result = chain.invoke({"text": chunks[i]})
    print(result.content)